# Set up

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import re, os, sqlite3
from datetime import datetime

# Function to extract score from options based on values in the response
def val_score_mapping(s1,s2):
    
    split_options = s2.strip().split("),")
    split_response = s1.strip().split(": ")[1].split(',')
    scores = {} 

    for i in split_options:
        if len(i) > 0:
            val_num = i.split("(score")[0].split(': ')[1].strip()
            score_num = i.split("(score")[1].split(': ')[1].strip()
            scores[val_num] = score_num

    response_score_mapping = {split_response[i].strip(): scores[split_response[i].strip()] for i in range(len(split_response))}
    list_response_score_mapping = list(response_score_mapping.values())
    str_response_score_mapping = ', '.join(str(value) for value in list_response_score_mapping)
    return str_response_score_mapping.replace(')', '')

# Function to cleanup and split time range in the response
def clean_time_range(df, column_name):
    cleaned = [] 
    for i in range(len(df[column_name])):
        if pd.notna(df[column_name][i]) and str(df[column_name][i]).startswith('time_range'):
            t = re.sub(r'[a-zA-Z\s+(\)_:]', '', df[column_name][i])
            t = t.replace(',', ':')
            if re.search(r'^[0-9]:', t): #9,30/12,30
                ttemp = '0' + t
            elif re.search(r':[0-9]$', t): #12,5/12,30
                ttemp = t.replace(':', ':0')
            else:
                ttemp = t
            # tpos = datetime.strptime(str(ttemp), '%H:%M')
            # thm = tpos.strftime('%H:%M')
            thm = ttemp      
        elif pd.notna(df[column_name][i]) and str(df[column_name][i]).startswith(' to'):
            t = re.sub(r'[a-zA-Z\s+(\)_:]', '', df[column_name][i])
            t = t.replace(',', ':')
            if re.search(r'^[0-9]:', t): #9,30/12,30
                ttemp = '0' + t
            elif re.search(r':[0-9]$', t): #12,5/12,30
                ttemp = t.replace(':', ':0')
            else:
                ttemp = t
            thm = ttemp      
        else:
            ttemp = df[column_name][i] 
            thm = ttemp
        cleaned.append(thm)
    return cleaned

#### Read in data

In [2]:
# Set up path variable and get list of responses.csv files

input_path = os.path.expanduser('~/NIMH EMA Data/Input Files/') #home directory > replace with your EMA folder > new folder called Input Files
all_files = os.listdir(os.path.join(input_path, 'EMA_applet_data'))
files = [file for file in all_files if file.startswith('responses')]
input_files = os.listdir(input_path)
output_path = os.path.expanduser('~/NIMH EMA Data/Output Files/')

# Read all the responses.csv files
responses_all = []
for i in range(len(files)):
    temp_df = pd.read_csv(os.path.join(input_path, 'EMA_applet_data', files[i]), encoding='ISO-8859-1')
    responses_all.append(temp_df)

# Concat responses.csv to one file
dat_full = pd.concat(responses_all, ignore_index=True)

dat_full = dat_full.map(str)

# Write out the concatenated responses.csv file
dat_full.to_csv(os.path.join(output_path,'responses_all.csv'),index=False)

# Data Cleaning

In [3]:
# Rename target_id column as it contains special characters
dat_full.rename(columns={'ï»¿target_id': 'target_id'}, inplace=True) 

# Add utc_timezone_offset
dat_full['offset'] = np.where(dat_full['utc_timezone_offset']=='nan', 0, 1)

dat_full['activity_start_time_offsetADD'] = np.where(dat_full['utc_timezone_offset'] != 'nan', pd.to_numeric(dat_full['activity_start_time'], errors='coerce')+ (pd.to_numeric(dat_full['utc_timezone_offset'], errors='coerce')* 60 * 1000) , pd.to_numeric(dat_full['activity_start_time'], errors='coerce'))
dat_full['activity_end_time_offsetADD'] = np.where(dat_full['utc_timezone_offset'] != 'nan', pd.to_numeric(dat_full['activity_end_time'], errors='coerce')+ (pd.to_numeric(dat_full['utc_timezone_offset'], errors='coerce')* 60 * 1000) , pd.to_numeric(dat_full['activity_end_time'], errors='coerce'))
dat_full['activity_schedule_start_time_offsetADD'] = np.where(dat_full['utc_timezone_offset'] != 'nan', pd.to_numeric(dat_full['activity_schedule_start_time'], errors='coerce')+(pd.to_numeric(dat_full['utc_timezone_offset'], errors='coerce')* 60 * 1000) , pd.to_numeric(dat_full['activity_schedule_start_time'], errors='coerce'))

dat_full['activity_start_time_offsetADD'] = pd.to_numeric(dat_full['activity_start_time_offsetADD'], downcast='integer')
dat_full['activity_end_time_offsetADD'] = pd.to_numeric(dat_full['activity_end_time_offsetADD'], downcast='integer')
dat_full['activity_schedule_start_time_offsetADD'] = pd.to_numeric(dat_full['activity_schedule_start_time_offsetADD'], downcast='integer')

dat_full['start_Time'] = dat_full['activity_start_time_offsetADD']
dat_full['end_Time'] = dat_full['activity_end_time_offsetADD']
dat_full['schedule_Time'] = dat_full['activity_schedule_start_time_offsetADD']

In [4]:
# Adjusting start and end times
dat_full = dat_full.groupby(['secret_user_id', 'activity_flow_id', 'activity_schedule_start_time'], group_keys=True).apply(lambda x: x.assign(start_Time=x['start_Time'].min(), end_Time=x['end_Time'].max())).reset_index(drop=True)


/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_62737/3553000024.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dat_full = dat_full.groupby(['secret_user_id', 'activity_flow_id', 'activity_schedule_start_time'], group_keys=True).apply(lambda x: x.assign(start_Time=x['start_Time'].min(), end_Time=x['end_Time'].max())).reset_index(drop=True)


In [6]:
# Employ cleaning code provided by Curious team

dat_subset = dat_full[['activity_submission_id', 'activity_schedule_start_time', 'secret_user_id', 'userId',
       'activity_id', 'activity_name', 'activity_flow_id', 'activity_flow_name', 'item_name', 'item_response', 'item_response_options',
       'activity_schedule_id', 'start_Time', 'end_Time', 'schedule_Time', 'applet_version', 'activity_start_time', 'offset']]

# Create additional column to add scores in the next steps
dat_subset = dat_subset.copy()
dat_subset['item_response_scores'] = None

# Cleanup
for i in range(len(dat_subset['item_response'])):
    if re.search(r'score: ', dat_subset['item_response_options'][i]):
        s = val_score_mapping(dat_subset['item_response'][i], dat_subset['item_response_options'][i])
        dat_subset.loc[i, 'item_response_scores'] = s  
    if re.search(r'value', dat_subset['item_response'][i]):
        r = dat_subset['item_response'][i].replace("value: ", "")
        dat_subset.loc[i, 'item_response'] = r
    elif re.search(r'time:', dat_subset['item_response'][i]):
        if re.search(r'hr [0-9],', dat_subset['item_response'][i]): 
            egapp = dat_subset['item_response'][i]. replace('time: hr ', '0')
            if re.search(r', min [0-9]$', egapp): 
                egtemp = egapp.replace(', min ', ':0')
            elif re.search(r', min [0-9][0-9]$', egapp): 
                egtemp = egapp.replace(', min ', ':')
            egpos = datetime.strptime(egtemp, '%H:%M')
            egpos2 = egpos.strftime('%H:%M')
            dat_subset.loc[i, 'item_response'] = egpos2
        elif re.search(r'hr [0-9][0-9],', dat_subset['item_response'][i]):
            egapp = dat_subset['item_response'][i].replace('time: hr ', '')
            if re.search(r', min [0-9]$', egapp): 
                egtemp = egapp.replace(', min ', ':0')
            elif re.search(r', min [0-9][0-9]$', egapp): 
                egtemp = egapp.replace(', min ', ':')
            egpos = datetime.strptime(egtemp, '%H:%M')
            egpos2 = egpos.strftime('%H:%M')
            dat_subset.loc[i, 'item_response'] = egpos2
    elif re.search(r'geo:', dat_subset['item_response'][i]):
        g = dat_subset['item_response'][i].replace('geo: ', '')
        dat_subset.loc[i, 'item_response'] = g

# Combine scores and other formats of responses into one column
dat_subset['item_response2'] = np.where(dat_subset['item_response_scores'].isna(), dat_subset['item_response'], dat_subset['item_response_scores'])

# Sort and Select required columns
dat_subset = dat_subset.sort_values(by=['secret_user_id', 'activity_flow_id', 'activity_id', 'activity_name', 'schedule_Time', 'activity_start_time'])

### Additional Cleaning

In [7]:
# Create a new binary to indicate whether responses were from activity or flow
dat_subset['is_activity'] = np.where(dat_subset['activity_flow_id'] == 'nan', 'Y', 'N')

# Create a new column such that:
#   activity_flow_id if item is from EMA
#   activity_id if item is from activity (device) log
dat_subset['activity_flow'] = np.where(dat_subset['activity_flow_id'] == 'nan', (dat_subset['activity_id'] + '|' + dat_subset['activity_name']), dat_subset['activity_flow_id'])
dat_subset = dat_subset[['userId', 'secret_user_id', 'activity_flow_id', 'activity_id', 'activity_flow', 'activity_flow_name', 'is_activity', 'offset', 'item_name', 'item_response', 
                         'item_response_scores', 'item_response2','item_response_options', 'start_Time',
                         'end_Time', 'schedule_Time', 'activity_schedule_start_time', 'applet_version', 'activity_submission_id', 'activity_schedule_id']]
 
# Make sure there are no NAs 
dat_subset['schedule_Time'] = np.where(dat_subset['schedule_Time'].isna(), 'NO SCHEDULE', dat_subset['schedule_Time'])

### Widening Data

In [8]:
# Add answer_ids to help with debugging in the final output
answers = dat_subset.groupby(['userId', 'secret_user_id', 'activity_flow', 'activity_flow_name', 'activity_schedule_id', 'is_activity', 'start_Time', 'end_Time', 'schedule_Time', 'offset', 'applet_version'])['activity_submission_id'].apply(lambda x: '|'.join(x.astype(str))).reset_index()

# Widen data
dat_wide = pd.pivot_table(dat_subset, index=['userId', 'secret_user_id', 'activity_flow', 'activity_flow_name', 'activity_schedule_id', 'is_activity', 'start_Time', 'end_Time', 'schedule_Time', 'offset', 'applet_version'], columns='item_name', values='item_response2', aggfunc='last').reset_index()

# Join Wide format table with answers to include concatenated answer_ids
dat_wide = pd.merge(dat_wide, answers, on=['userId', 'secret_user_id', 'activity_flow', 'activity_flow_name', 'activity_schedule_id', 'is_activity', 'start_Time', 'end_Time', 'schedule_Time', 'offset', 'applet_version'], how='outer')

dat_wide.head()

,userId,secret_user_id,activity_flow,activity_flow_name,activity_schedule_id,is_activity,start_Time,end_Time,schedule_Time,offset,...,since_pain,since_pain_where,since_physical_activity,since_sedentary,since_times_eat,socialmedia_activity,socialmedia_duration,strangers_communication_method,videogame_duration,activity_submission_id
0,a4314838-bb7f-49d1-9617-b0bdff307e36,9999-999,5c7ffe36-3870-425f-9b03-2a84672bf4d7,Evening Assessment,35c1c40b-3f99-4cfc-be14-ec0c59f18d52,N,1776285351944,1776285557960,1776283200000.0,1,...,1,"4, 3","1, 3",NaN,2,NaN,NaN,NaN,NaN,db583ddc-3e39-4e94-b9ee-28983f9dcf78|db583ddc-...
1,a4314838-bb7f-49d1-9617-b0bdff307e36,9999-999,6cf0cbb3-7ba4-46c9-93f1-e2978774d607,Morning Assessment,32731910-98ab-49c2-ad99-ec647b8d1d7e,N,1776327177079,1776327324637,1776326400000.0,1,...,0,NaN,"2, 4",1,0,"3, 1, 7",3,1,NaN,f33f4f6a-5a9f-45a0-80d7-8dff00e3ead6|f33f4f6a-...
2,a4314838-bb7f-49d1-9617-b0bdff307e36,9999-999,90fdafd8-d9de-4a7c-954d-f721296965ac|Activity ...,nan,8d5972da-3586-4b27-b0f5-b200904cc7b4,Y,1779280539799,1779280544021,NO SCHEDULE,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9224387b-bdc4-4a94-ae39-f6fc1c8895f1
3,a4314838-bb7f-49d1-9617-b0bdff307e36,9999-999,b41f601c-30df-42bd-9e2b-acb610bf5123,Mid-day Assessment,50720872-2cbc-42d0-a1f4-7da5ebabaaa4,N,1776340876322,1776341089903,1776340800000.0,1,...,1,"2, 1","1, 4",2,2,"5, 6",2,NaN,1,b32658c4-a95c-4e4d-a40e-44a65c147919|b32658c4-...
4,a4314838-bb7f-49d1-9617-b0bdff307e36,9999-999,caf8ef2b-bfe8-417c-9c2f-cdc9d61a94e3,Afternoon Assessment,8633fb32-e210-4abc-9848-ecf983ade894,N,1776268833078,1776268972369,1776268800000.0,1,...,0,NaN,"2, 4",3,10,NaN,NaN,NaN,1,975b5363-cf29-490c-802f-f0a865555510|975b5363-...


### Specific item cleaning

In [9]:
# Use the function from the Set Up section to split headache_time_range column into start and end times
headache_times = ['headache_time_start', 'headache_time_end']

for i in headache_times: 
    if i in dat_wide.columns:
        dat_wide[i] = clean_time_range(dat_wide, i)
        dat_wide[i] = clean_time_range(dat_wide, i)

# Clean up gps response and split to 2 columns for latitude and longitude
if 'now_gps' in dat_wide.columns:
    dat_wide['now_gps'] = dat_wide['now_gps'].replace(r'[a-zA-Z\s+(\)]', '', regex = True)
    dat_wide[['now_gps_lat', 'now_gps_long']] = dat_wide['now_gps'].str.split("/", expand=True)

# Use the function from the Set Up section to split since_activity_monitor_time and since_light_device_time columns into start and end times
devices_time_range = ['since_activity_monitor_time', 'since_light_device_time']

for i in devices_time_range: 
    if i in dat_wide.columns:
        start = i + '_start'
        end = i + '_end'
        dat_wide[[start, end]] = dat_wide[i].str.split("/", expand=True)
        dat_wide[start] = clean_time_range(dat_wide, start)
        dat_wide[end] = clean_time_range(dat_wide, end) 

### Timestamp cleaning

In [10]:
dat_wide_full = dat_wide.map(str)

# Epoch to Timestamp
dat_wide_full['start_Time'] = pd.to_numeric(dat_wide_full['start_Time'], errors='coerce')
dat_wide_full['start_Time'] = pd.to_datetime(dat_wide_full['start_Time'] / 1000, unit='s')

dat_wide_full['end_Time'] = pd.to_numeric(dat_wide_full['end_Time'], errors='coerce')
dat_wide_full['end_Time'] = pd.to_datetime(dat_wide_full['end_Time'] / 1000, unit='s')

dat_wide_full['schedule_Time'] = pd.to_numeric(dat_wide_full['schedule_Time'], errors='coerce')
dat_wide_full['schedule_Time'] = pd.to_datetime(dat_wide_full['schedule_Time'] / 1000, unit='s')

# Timestamp cleanup 
dat_wide_full['schedule_Time'] = pd.to_datetime(dat_wide_full['schedule_Time'])
dat_wide_full['schedule_Time'] = np.where(dat_wide_full['offset']== '1', dat_wide_full['schedule_Time'],  dat_wide_full['schedule_Time'].dt.tz_localize('UTC').dt.tz_convert('America/New_York').dt.tz_localize(None)) 

dat_wide_full['start_Time'] = pd.to_datetime(dat_wide_full['start_Time'])
dat_wide_full['start_Time'] = dat_wide_full['start_Time'].dt.floor('1s')
dat_wide_full['start_Time'] = np.where(dat_wide_full['offset']== '1', dat_wide_full['start_Time'], dat_wide_full['start_Time'].dt.tz_localize('UTC').dt.tz_convert('America/New_York').dt.tz_localize(None))

dat_wide_full['end_Time'] = pd.to_datetime(dat_wide_full['end_Time'])
dat_wide_full['end_Time'] = dat_wide_full['end_Time'].dt.floor('1s')
dat_wide_full['end_Time'] = np.where(dat_wide_full['offset']== '1', dat_wide_full['end_Time'], dat_wide_full['end_Time'].dt.tz_localize('UTC').dt.tz_convert('America/New_York').dt.tz_localize(None))

# Creating Dates
dat_wide_full['scheduled_Date'] = dat_wide_full['schedule_Time'].dt.date
dat_wide_full['start_Date'] = dat_wide_full['start_Time'].dt.date

### Split data into flows and activities

In [11]:
act_dat_wide = dat_wide_full[dat_wide_full['is_activity']=='Y'] # activity (device) data
flow_dat_wide = dat_wide_full[dat_wide_full['is_activity']=='N'] # flow (EMA) data

# Flow Submissions (EMA only)

In [12]:
# Timestamp formatting - TO MAKE SURE!
flow_dat_wide['schedule_Time'] = pd.to_datetime(flow_dat_wide['schedule_Time'], format='mixed')
flow_dat_wide['start_Time'] = pd.to_datetime(flow_dat_wide['start_Time'], format="mixed")
flow_dat_wide['end_Time'] = pd.to_datetime(flow_dat_wide['end_Time'])

final = flow_dat_wide

/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_62737/1239275528.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  flow_dat_wide['schedule_Time'] = pd.to_datetime(flow_dat_wide['schedule_Time'], format='mixed')
/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_62737/1239275528.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  flow_dat_wide['start_Time'] = pd.to_datetime(flow_dat_wide['start_Time'], format="mixed")
/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_62737/12392

### Flows Final

In [ ]:
#################################################################### PLEASE SELECT APPROPRIATE ITEMS & ORDER ###############################################################

# Specify the required columns, in desired order, for the final output

cols = [
    'userId',
    'activity_schedule_id',
    'activity_submission_id',
    'secret_user_id',
    'activity_flow_name',
    'schedule_Time',
    'start_Time',
    'end_Time',
    'applet_version',
    'since_activity_monitor',
    'since_light_device',
    'morning_sleep_quantity',
    'morning_bedtime',
    'morning_wake_number',
    'morning_waketime',
    'morning_sleep_quality',
    'morning_sleep_refreshed',
    'morning_sleep_problems',
    'now_gps_lat', 
    'now_gps_long', 
    'now_where',
    'now_inside',
    'now_outside',
    'now_company',
    'now_activity',
    'since_internet',
    'internet_use_duration',
    'internet_use_category',
    'ai_chatbot_duration',
    'ai_chatbot_category',
    'ai_chatbot_category_other',
    'videogame_duration',
    'socialmedia_duration',
    'socialmedia_activity',
    'friends_communication_method',
    'strangers_communication_method',
    'comments_byothers',
    'comments_byself',
    'now_sadness',
    'now_anxiousness',
    'now_active',
    'now_tired',
    'now_distracted',
    'now_irritable',
    'now_quick_thinking',
    'now_enjoyment',
    'now_fidgety',
    'now_thoughts_positive',
    'now_thoughts_negative',
    'now_thoughts_negative_about',
    'now_thoughts_negative_severity',
    'since_thoughts_negative',
    'since_thoughts_negative_suicide',
    'since_thoughts_negative_suicide_message',
    'instructions', #event_instructions
    'event_emotion',
    'event_category',
    'event_people',
    'event_where',
    'event_health',
    'event_content',
    'event_issue',
    'event_stress',
    'since_physical_activity',
    'since_sedentary',
    'since_rest_fell_asleep',
    'now_thirsty',
    'since_had_drink',
    'not_alcohol_amount',
    'since_had_drink_caffeinated_type',
    'now_hungry',
    'since_times_eat',
    'since_eaten_amount',
    'since_eaten_type',
    'since_food1',
    'since_food2',
    'since_food3',
    'since_food4',
    'since_food5',
    'since_crave',
    'now_pain',
    'now_pain_where',
    'since_pain',
    'since_pain_where',
    'pain_intensity',
    'headache',
    'headache_intensity',
    'headache_location',
    'headache_nausea',
    'headache_light',
    'headache_heat',
    'headache_noise',
    'headache_smell',
    'headache_trigger',
    'day_physical_health',
    'day_problem_categories',
    'day_bother',
    'day_problems_allergies',
    'day_problems_breath',
    'day_problems_belly_symptoms',
    'day_problems_belly',
    'day_problems_muscle',
    'day_problems_heart',
    'dizziness_faint',
    'day_period'
    ]

# Select and arrange columns, adding manual cols as NA if needed (to ensure all variables are displayed)
final = final.reindex(columns=cols)

# Sort by user ID and time of assessment
final = final.sort_values(by=['secret_user_id', 'schedule_Time', 'start_Time'])

In [14]:
# Ensure consistent NA wording across all columns
final_out = final.copy()
final_out.replace('nan', 'NA', inplace=True)
final_out.fillna('NA', inplace=True)
final_out.replace('NA NA', 'NA', inplace=True)
final_out.rename(columns = {'userId_x':'user_id'},inplace=True)

# Exclude test flows
final_out = final_out[final_out['activity_flow_name'] != 'Test Flow (All Activities)']

# Check final flow data
final_out.head()

/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_62737/729725924.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NA' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  final_out.fillna('NA', inplace=True)


,userId,activity_schedule_id,activity_submission_id,secret_user_id,activity_flow_name,schedule_Time,start_Time,end_Time,applet_version,since_activity_monitor,...,day_problem_categories,day_bother,day_problems_allergies,day_problems_breath,day_problems_belly_symptoms,day_problems_belly,day_problems_muscle,day_problems_heart,dizziness_faint,day_period
4,a4314838-bb7f-49d1-9617-b0bdff307e36,8633fb32-e210-4abc-9848-ecf983ade894,975b5363-cf29-490c-802f-f0a865555510|975b5363-...,9999-999,Afternoon Assessment,2026-04-15 16:00:00,2026-04-15 16:00:33,2026-04-15 16:02:52,1.1.0,0,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
0,a4314838-bb7f-49d1-9617-b0bdff307e36,35c1c40b-3f99-4cfc-be14-ec0c59f18d52,db583ddc-3e39-4e94-b9ee-28983f9dcf78|db583ddc-...,9999-999,Evening Assessment,2026-04-15 20:00:00,2026-04-15 20:35:51,2026-04-15 20:39:17,1.1.0,1,...,"4, 5, 6",3,NA,NA,NA,NA,1,3,1,0
1,a4314838-bb7f-49d1-9617-b0bdff307e36,32731910-98ab-49c2-ad99-ec647b8d1d7e,f33f4f6a-5a9f-45a0-80d7-8dff00e3ead6|f33f4f6a-...,9999-999,Morning Assessment,2026-04-16 08:00:00,2026-04-16 08:12:57,2026-04-16 08:15:24,1.1.0,0,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
3,a4314838-bb7f-49d1-9617-b0bdff307e36,50720872-2cbc-42d0-a1f4-7da5ebabaaa4,b32658c4-a95c-4e4d-a40e-44a65c147919|b32658c4-...,9999-999,Mid-day Assessment,2026-04-16 12:00:00,2026-04-16 12:01:16,2026-04-16 12:04:49,1.1.0,1,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [15]:
# Write out final flow dataframe: necessary for secondary QC script.

final_out.to_csv(os.path.join(output_path, 'flow_final.csv'), index=False, date_format='%Y-%m-%d %H:%M:%S', na_rep='NA')

# Activity Submissions (devices only)

In [16]:
# Check your ACTIVITIES Submissions data:
# Will only have data from activities like activity watch or light sensors logs, otherwise dataset below will be empty. 
# If empty, you can skip the rest of the script.

act_dat_wide.head()

,userId,secret_user_id,activity_flow,activity_flow_name,activity_schedule_id,is_activity,start_Time,end_Time,schedule_Time,offset,...,since_times_eat,socialmedia_activity,socialmedia_duration,strangers_communication_method,videogame_duration,activity_submission_id,now_gps_lat,now_gps_long,scheduled_Date,start_Date
2,a4314838-bb7f-49d1-9617-b0bdff307e36,9999-999,90fdafd8-d9de-4a7c-954d-f721296965ac|Activity ...,nan,8d5972da-3586-4b27-b0f5-b200904cc7b4,Y,2026-05-20 12:35:39,2026-05-20 12:35:44,NaT,1,...,nan,nan,nan,nan,nan,9224387b-bdc4-4a94-ae39-f6fc1c8895f1,nan,nan,NaT,2026-05-20


In [17]:
# Make schedule_Time = start_Time (since they do not have scheduled times)
for index, row in act_dat_wide.iterrows():
    act_dat_wide.at[index, 'schedule_Time'] = row['start_Time']

In [18]:
# Timestamp formatting - TO MAKE SURE!
act_dat_wide['schedule_Time'] = pd.to_datetime(act_dat_wide['schedule_Time'], format="mixed")
act_dat_wide['start_Time'] = pd.to_datetime(act_dat_wide['start_Time'], format = "mixed")
act_dat_wide['end_Time'] = pd.to_datetime(act_dat_wide['end_Time'], format = "mixed")

act_final = act_dat_wide

/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_62737/3808806946.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  act_dat_wide['schedule_Time'] = pd.to_datetime(act_dat_wide['schedule_Time'], format="mixed")
/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_62737/3808806946.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  act_dat_wide['start_Time'] = pd.to_datetime(act_dat_wide['start_Time'], format = "mixed")
/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_62737/3808806

### Activities Final

In [19]:
################################################################################### PLEASE SELECT APPROPRIATE ITEMS & ORDER ############################################################################

# Uses the same "cols" list provided above to specify the required columns, in desired order, for the final output

# Select and arrange columns, adding manual cols as NA if needed (to ensure all variables are displayed)
act_final = act_final.reindex(columns=cols)

# Sort by user ID and time of assessment
act_final = act_final.sort_values(by=['secret_user_id', 'start_Time'])

In [20]:
# Ensure consistent NA wording across all columns
act_final_out = act_final.copy()
act_final_out.replace('nan', 'NA', inplace=True)
act_final_out.fillna('NA', inplace=True)
act_final_out.replace('NA NA', 'NA', inplace=True)

# Check final activity data
act_final_out.head()

/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_62737/3480809165.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NA' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  act_final_out.fillna('NA', inplace=True)


,userId,activity_schedule_id,activity_submission_id,secret_user_id,activity_flow_name,schedule_Time,start_Time,end_Time,applet_version,since_activity_monitor,...,day_problem_categories,day_bother,day_problems_allergies,day_problems_breath,day_problems_belly_symptoms,day_problems_belly,day_problems_muscle,day_problems_heart,dizziness_faint,day_period
2,a4314838-bb7f-49d1-9617-b0bdff307e36,8d5972da-3586-4b27-b0f5-b200904cc7b4,9224387b-bdc4-4a94-ae39-f6fc1c8895f1,9999-999,NA,2026-05-20 12:35:39,2026-05-20 12:35:39,2026-05-20 12:35:44,1.1.1,1,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [21]:
# Write out final activity dataframe. Again, will only have data from activities like watch/sensor logs
act_final_out.to_csv(os.path.join(output_path, 'activity_final.csv'), index=False, date_format='%Y-%m-%d %H:%M:%S', na_rep='NA')

# Final Output (flow and activity together)

In [22]:
# Join flow final and activity final togeter 
# If you don't have any activity data, this will be identical to flow final and you can skip the rest of the script.

all_final = [final_out, act_final_out]
ema_out = pd.concat(all_final, ignore_index=True)

# Ensure alignment with formatting
ema_out = ema_out.applymap(str)

# Final data output sorting
ema_out = ema_out.sort_values(by=['secret_user_id', 'schedule_Time'])

/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_62737/1900246699.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  ema_out = ema_out.applymap(str)


Check final output

In [23]:
# Check for duplicates in final file. Ensure this always returns "False"
print(ema_out.duplicated().any())

False


In [24]:
# Preview final output
ema_out.head()

,userId,activity_schedule_id,activity_submission_id,secret_user_id,activity_flow_name,schedule_Time,start_Time,end_Time,applet_version,since_activity_monitor,...,day_problem_categories,day_bother,day_problems_allergies,day_problems_breath,day_problems_belly_symptoms,day_problems_belly,day_problems_muscle,day_problems_heart,dizziness_faint,day_period
0,a4314838-bb7f-49d1-9617-b0bdff307e36,8633fb32-e210-4abc-9848-ecf983ade894,975b5363-cf29-490c-802f-f0a865555510|975b5363-...,9999-999,Afternoon Assessment,2026-04-15 16:00:00,2026-04-15 16:00:33,2026-04-15 16:02:52,1.1.0,0,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
1,a4314838-bb7f-49d1-9617-b0bdff307e36,35c1c40b-3f99-4cfc-be14-ec0c59f18d52,db583ddc-3e39-4e94-b9ee-28983f9dcf78|db583ddc-...,9999-999,Evening Assessment,2026-04-15 20:00:00,2026-04-15 20:35:51,2026-04-15 20:39:17,1.1.0,1,...,"4, 5, 6",3,NA,NA,NA,NA,1,3,1,0
2,a4314838-bb7f-49d1-9617-b0bdff307e36,32731910-98ab-49c2-ad99-ec647b8d1d7e,f33f4f6a-5a9f-45a0-80d7-8dff00e3ead6|f33f4f6a-...,9999-999,Morning Assessment,2026-04-16 08:00:00,2026-04-16 08:12:57,2026-04-16 08:15:24,1.1.0,0,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
3,a4314838-bb7f-49d1-9617-b0bdff307e36,50720872-2cbc-42d0-a1f4-7da5ebabaaa4,b32658c4-a95c-4e4d-a40e-44a65c147919|b32658c4-...,9999-999,Mid-day Assessment,2026-04-16 12:00:00,2026-04-16 12:01:16,2026-04-16 12:04:49,1.1.0,1,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
4,a4314838-bb7f-49d1-9617-b0bdff307e36,8d5972da-3586-4b27-b0f5-b200904cc7b4,9224387b-bdc4-4a94-ae39-f6fc1c8895f1,9999-999,NA,2026-05-20 12:35:39,2026-05-20 12:35:39,2026-05-20 12:35:44,1.1.1,1,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [25]:
# Write the final output file to csv
ema_out.to_csv(os.path.join(output_path, 'ema_output_final.csv'), index=False, date_format='%Y-%m-%d %H:%M:%S', na_rep='NA')

In [ ]:
# For a specific participant:
# xxx_ema = ema_out[ema_out['secret_user_id'] == '9999-999']
# xxx_ema.to_csv(os.path.join(output_path, '9999-999_ema_output_final.csv'), index=False, date_format='%Y-%m-%d %H:%M:%S', na_rep='NA')